# Reference Trainer Baseline — ResNet-20 on CIFAR-10

## Purpose
This notebook establishes an upper-bound synchronous baseline by faithfully replicating the training recipe from the reference implementation by **Yerlan Idelbayev**:

> Idelbayev, Y. *Proper ResNets for CIFAR-10 in PyTorch*. GitHub: [akamaster/pytorch_resnet_cifar10](https://github.com/akamaster/pytorch_resnet_cifar10)

The reference reports **~91.25% top-1 accuracy** for ResNet-20 on CIFAR-10.

## How this differs from `train_baseline.ipynb`
| Aspect | `train_baseline.ipynb` | This notebook |
|---|---|---|
| Momentum | 0.0 (vanilla SGD) | **0.9** |
| Epochs | 164 | **200** |
| LR milestones | [82, 123] | **[100, 150]** |
| Normalization | CIFAR-10 stats | **CIFAR-10 stats** |
| Per-batch metrics | No | **Yes** (AverageMeter) |

The momentum and learning-rate schedule are the primary drivers of the improved accuracy.

In [1]:
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from tqdm.notebook import tqdm

sys.path.append(str(Path.cwd().parent))
from src.model import resnet20

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cudnn.benchmark = True
device

device(type='cuda')

In [3]:
class AverageMeter:
    """Computes and stores the average and current value."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def accuracy(output, target, topk=(1,)):
    """Computes precision@k for the specified values of k."""
    maxk = max(topk)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    return [
        correct[:k].reshape(-1).float().sum(0).mul_(100.0 / batch_size)
        for k in topk
    ]

In [ ]:
# CIFAR-10 normalization statistics
normalize = transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                                 std=[0.2023, 0.1994, 0.2010])

DATA_ROOT = "../../data"

train_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        root=DATA_ROOT, train=True, download=True,
        transform=transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            normalize,
        ])
    ),
    batch_size=128, shuffle=True, num_workers=4, pin_memory=True
)

val_loader = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=False,
        transform=transforms.Compose([
            transforms.ToTensor(),
            normalize,
        ])
    ),
    batch_size=128, shuffle=False, num_workers=4, pin_memory=True
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [5]:
model = resnet20().to(device)

criterion = nn.CrossEntropyLoss().to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[100, 150],
    gamma=0.1
)

In [6]:
PRINT_FREQ = 50
NUM_EPOCHS = 200
best_prec1 = 0.0

for epoch in tqdm(range(NUM_EPOCHS), desc="Epochs"):
    # --- Train ---
    model.train()
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  lr={optimizer.param_groups[0]['lr']:.5e}")
    end = time.time()

    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        prec1 = accuracy(outputs.detach(), targets)[0]
        losses.update(loss.item(), inputs.size(0))
        top1.update(prec1.item(), inputs.size(0))
        batch_time.update(time.time() - end)
        end = time.time()

        if i % PRINT_FREQ == 0:
            print(f"  [{i}/{len(train_loader)}]  "
                  f"Time {batch_time.val:.3f} ({batch_time.avg:.3f})  "
                  f"Loss {losses.val:.4f} ({losses.avg:.4f})  "
                  f"Prec@1 {top1.val:.3f} ({top1.avg:.3f})")

    scheduler.step()

    # --- Validate ---
    model.eval()
    val_losses = AverageMeter()
    val_top1 = AverageMeter()

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            prec1 = accuracy(outputs, targets)[0]
            val_losses.update(loss.item(), inputs.size(0))
            val_top1.update(prec1.item(), inputs.size(0))

    best_prec1 = max(best_prec1, val_top1.avg)
    print(f"  Val Loss: {val_losses.avg:.4f}  Prec@1: {val_top1.avg:.3f}  Best: {best_prec1:.3f}")

Epochs:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 1/200  lr=1.00000e-01
  [0/391]  Time 1.096 (1.096)  Loss 4.2821 (4.2821)  Prec@1 10.156 (10.156)
  [50/391]  Time 0.021 (0.038)  Loss 1.8837 (2.3236)  Prec@1 21.875 (19.393)
  [100/391]  Time 0.015 (0.028)  Loss 1.7783 (2.0986)  Prec@1 26.562 (23.198)
  [150/391]  Time 0.016 (0.024)  Loss 1.6184 (1.9798)  Prec@1 38.281 (26.588)
  [200/391]  Time 0.022 (0.023)  Loss 1.6370 (1.8990)  Prec@1 39.844 (29.062)
  [250/391]  Time 0.018 (0.022)  Loss 1.5750 (1.8369)  Prec@1 42.969 (31.505)
  [300/391]  Time 0.016 (0.021)  Loss 1.5570 (1.7811)  Prec@1 48.438 (33.672)
  [350/391]  Time 0.018 (0.021)  Loss 1.2542 (1.7330)  Prec@1 53.125 (35.470)
  Val Loss: 1.3969  Prec@1: 48.270  Best: 48.270
Epoch 2/200  lr=1.00000e-01
  [0/391]  Time 0.210 (0.210)  Loss 1.2314 (1.2314)  Prec@1 57.031 (57.031)
  [50/391]  Time 0.013 (0.022)  Loss 1.1747 (1.2841)  Prec@1 54.688 (53.048)
  [100/391]  Time 0.012 (0.020)  Loss 1.2847 (1.2647)  Prec@1 60.156 (53.752)
  [150/391]  Time 0.016 (0.020)  Loss 1.162

In [7]:
print(f"Training complete.")
print(f"Best Prec@1: {best_prec1:.2f}%")

Training complete.
Best Prec@1: 91.72%


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(targets.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds) * 100
precision = precision_score(all_labels, all_preds, average=None, zero_division=0) * 100
recall    = recall_score(all_labels, all_preds, average=None, zero_division=0) * 100
f1        = f1_score(all_labels, all_preds, average=None, zero_division=0) * 100

print(f"Overall Accuracy : {acc:.2f}%")
print(f"Macro Precision  : {precision.mean():.2f}%")
print(f"Macro Recall     : {recall.mean():.2f}%")
print(f"Macro F1-Score   : {f1.mean():.2f}%\n")
print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 46)
for i, cls in enumerate(CIFAR10_CLASSES):
    print(f"{cls:<12} {precision[i]:>9.2f}% {recall[i]:>9.2f}% {f1[i]:>9.2f}%")

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(CIFAR10_CLASSES))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width, precision, width, label='Precision',  color='steelblue')
bars2 = ax.bar(x,          recall,   width, label='Recall',     color='coral')
bars3 = ax.bar(x + width,  f1,       width, label='F1-Score',   color='mediumseagreen')

ax.axhline(acc, color='gray', linestyle='--', linewidth=1.2, label=f'Overall Accuracy ({acc:.1f}%)')

ax.set_xlabel('Class')
ax.set_ylabel('Score (%)')
ax.set_title('Per-Class Precision, Recall & F1-Score — ResNet-20 on CIFAR-10 (Reference Baseline)')
ax.set_xticks(x)
ax.set_xticklabels(CIFAR10_CLASSES, rotation=30, ha='right')
ax.set_ylim(0, 110)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()